# 02 | Regras de armas registradas pela RAND

Preparamos uma contagem de registros de regras em vigor no fim de cada ano, de 2019 a 2024. Mantemos também as colunas antigas sobre porte e antecedentes para preservar o experimento anterior de previsão.

**Como usar:** execute os três blocos de código, na ordem. Cada `Law ID` representa um registro da RAND, não necessariamente uma lei distinta.

## Bloco 1: Abrir e conhecer

A planilha oficial RAND já está na pasta `reports/attachments/`. Porte de arma escondida é chamado `CCW` na planilha.

In [4]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import pandas as pd
arquivo = ROOT / "reports/attachments/TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
leis = pd.read_excel(arquivo, sheet_name="Database")
porte = leis[leis["Law Class"].eq("carrying a concealed weapon (ccw)")].copy()
privadas = leis[(leis["Law Class"] == "background checks") & (leis["Law Class Subtype"] == "private sales : point of sale")].copy()
print("Registros de CCW:", len(porte), "| Antecedentes em vendas privadas:", len(privadas))

Registros de CCW: 142 | Antecedentes em vendas privadas: 46


## Bloco 2: Criar as colunas

Para cada estado e ano, consultamos as categorias da RAND ainda não substituídas. `may issue`: decisão da autoridade; `shall issue`: licença se cumpre requisitos; `permitless`: sem exigência de licença nessa modalidade.

In [5]:
registros_ccw = []
for estado in sorted(porte["State"].unique()):
    linhas = porte[porte["State"] == estado]
    for ano in range(2019, 2025):
        vigentes = linhas[(linhas["Effective Date Year"] <= ano) & (linhas["Supersession Date Year"].isna() | (linhas["Supersession Date Year"] > ano))]
        tipos = set(vigentes["Law Class Subtype"])
        registros_ccw.append({"state": estado, "year": ano,
            "ccw_shall_issue": int("shall issue" in tipos),
            "ccw_permitless": int("shall issue (permit not required)" in tipos),
            "ccw_may_issue": int("may issue" in tipos),
            "ccw_prohibited": int("prohibited" in tipos)})
ccw = pd.DataFrame(registros_ccw).query("state != 'District of Columbia'").copy()
registros_bg = []
for estado in sorted(ccw["state"].unique()):
    linhas = privadas[privadas["State"] == estado]
    for ano in range(2019, 2025):
        vigentes = linhas[(linhas["Effective Date Year"] <= ano) & (linhas["Supersession Date Year"].isna() | (linhas["Supersession Date Year"] > ano)) & (linhas["Type of Change"] != "Repeal")]
        tipos = set(vigentes["Handguns or Long Guns"])
        registros_bg.append({"state": estado, "year": ano,
            "background_handgun": int(bool(tipos & {"handgun", "handgun and long gun"})),
            "background_longgun": int(bool(tipos & {"long gun", "handgun and long gun"}))})
background = pd.DataFrame(registros_bg)
print("CCW:", ccw.shape, "| Antecedentes:", background.shape)

CCW: (300, 6) | Antecedentes: (300, 4)


## Bloco 3: Contar, conferir e salvar

Para a pergunta principal, contamos registros `Implement` ou `Modify` vigentes em **31 de dezembro** de cada ano. Separamos os que a RAND classifica como `Restrictive` e `Permissive`. `Repeal` e `See note` não entram nessa contagem. A classificação e a data de referência são decisões metodológicas que precisam acompanhar a interpretação dos resultados.

In [6]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.association_data import count_rand_records

for nome, tabela in [("CCW", ccw), ("Background", background)]:
    assert tabela.shape[0] == 300, f"{nome}: linhas inesperadas"
    assert tabela["state"].nunique() == 50
    assert not tabela.duplicated(["state", "year"]).any()
    destino = ROOT / ("data/processed/rand_ccw_2019_2024.csv" if nome == "CCW" else "data/processed/rand_background_2019_2024.csv")
    destino.parent.mkdir(parents=True, exist_ok=True)
    tabela.to_csv(destino, index=False)
    print(nome, "salvo:", destino.relative_to(ROOT))

estados = set(ccw["state"])
contagem = count_rand_records(leis, estados)
assert contagem.shape == (300, 5)
assert not contagem.duplicated(["state", "year"]).any()
destino = ROOT / "data/processed/rand_law_counts_2019_2024.csv"
contagem.to_csv(destino, index=False)
print("Contagem salva:", destino.relative_to(ROOT))
display(contagem.head())

CCW salvo: data/processed/rand_ccw_2019_2024.csv
Background salvo: data/processed/rand_background_2019_2024.csv
Contagem salva: data/processed/rand_law_counts_2019_2024.csv


,state,year,restrictive_records,permissive_records,total_law_records
0,Alabama,2019,11,3,14
1,Alabama,2020,11,3,14
2,Alabama,2021,11,3,14
3,Alabama,2022,11,3,14
4,Alabama,2023,11,3,14
